# Git Module: Git Workflow Simulator

**What you'll learn:**
- The mental model of git (working directory → staging → commits)
- How branches work
- What happens during a merge
- Visual understanding of git concepts

**Important:** This is a SIMULATION, not actual git. The goal is to build intuition about how git works.

**→ Run each cell and tell Claude what you see!**

In [1]:
# 🔧 SETUP - Run this first!

class GitSimulator:
    """A simplified git simulator for learning."""

    def __init__(self):
        self.working_dir = {}      # filename: content
        self.staging_area = {}     # filename: content
        self.commits = []          # list of {hash, message, files}
        self.branches = {"main": None}  # branch_name: commit_hash
        self.current_branch = "main"
        self.HEAD = None

    def status(self):
        """Show current state."""
        print(f"\n📍 Branch: {self.current_branch}")
        print(f"📌 HEAD: {self.HEAD[:7] if self.HEAD else 'None'}")
        print("\n📁 Working Directory:")
        for f, c in self.working_dir.items():
            staged = "[staged]" if f in self.staging_area else ""
            print(f"   {f}: {c[:30]}... {staged}")
        if not self.working_dir:
            print("   (empty)")
        print("\n📋 Staging Area:")
        for f, c in self.staging_area.items():
            print(f"   {f}: {c[:30]}...")
        if not self.staging_area:
            print("   (empty)")
        print(f"\n📜 Commits: {len(self.commits)}")

    def create_file(self, filename, content):
        """Create or modify a file in working directory."""
        self.working_dir[filename] = content
        print(f"✓ Created/modified: {filename}")

    def add(self, filename):
        """Stage a file for commit."""
        if filename not in self.working_dir:
            print(f"✗ File not found: {filename}")
            return
        self.staging_area[filename] = self.working_dir[filename]
        print(f"✓ Staged: {filename}")

    def add_all(self):
        """Stage all files."""
        for f in self.working_dir:
            self.staging_area[f] = self.working_dir[f]
        print(f"✓ Staged all {len(self.staging_area)} files")

    def commit(self, message):
        """Create a commit from staged files."""
        if not self.staging_area:
            print("✗ Nothing staged to commit")
            return
        import hashlib
        import time
        content = str(self.staging_area) + str(time.time())
        commit_hash = hashlib.sha1(content.encode()).hexdigest()
        commit = {
            "hash": commit_hash,
            "message": message,
            "files": dict(self.staging_area),
            "parent": self.HEAD
        }
        self.commits.append(commit)
        self.HEAD = commit_hash
        self.branches[self.current_branch] = commit_hash
        self.staging_area = {}
        print(f"✓ Committed: [{commit_hash[:7]}] {message}")

    def log(self):
        """Show commit history."""
        print("\n📜 Commit History:")
        print("─" * 50)
        for c in reversed(self.commits):
            marker = " (HEAD)" if c["hash"] == self.HEAD else ""
            print(f"  {c['hash'][:7]}{marker}")
            print(f"  └─ {c['message']}")
            print(f"     Files: {list(c['files'].keys())}")
            print()
        if not self.commits:
            print("  (no commits yet)")

    def branch(self, name):
        """Create a new branch."""
        self.branches[name] = self.HEAD
        print(f"✓ Created branch: {name}")

    def checkout(self, branch_name):
        """Switch to a branch."""
        if branch_name not in self.branches:
            print(f"✗ Branch not found: {branch_name}")
            return
        self.current_branch = branch_name
        self.HEAD = self.branches[branch_name]
        # Load files from that commit
        if self.HEAD:
            commit = next(c for c in self.commits if c["hash"] == self.HEAD)
            self.working_dir = dict(commit["files"])
        print(f"✓ Switched to branch: {branch_name}")

    def list_branches(self):
        """Show all branches."""
        print("\n🌿 Branches:")
        for b, h in self.branches.items():
            marker = " *" if b == self.current_branch else ""
            hash_str = h[:7] if h else "(no commits)"
            print(f"  {b}{marker} → {hash_str}")

    def visualize(self):
        """ASCII visualization of git state."""
        print("\n" + "═" * 60)
        print("                    GIT VISUALIZATION")
        print("═" * 60)

        # Working Directory
        print("\n┌─────────────────────────────────────────────────────┐")
        print("│              📁 WORKING DIRECTORY                   │")
        print("│  (Your actual files on disk)                        │")
        for f in self.working_dir:
            status = "[modified]" if f not in self.staging_area else "[staged]"
            print(f"│    • {f} {status:>20}                │"[:55] + "│")
        if not self.working_dir:
            print("│    (empty)                                          │")
        print("└────────────────────────┬────────────────────────────┘")
        print("                         │")
        print("                    git add")
        print("                         ▼")

        # Staging Area
        print("┌─────────────────────────────────────────────────────┐")
        print("│               📋 STAGING AREA                       │")
        print("│  (Files ready to be committed)                      │")
        for f in self.staging_area:
            print(f"│    • {f:45}│")
        if not self.staging_area:
            print("│    (empty)                                          │")
        print("└────────────────────────┬────────────────────────────┘")
        print("                         │")
        print("                   git commit")
        print("                         ▼")

        # Repository
        print("┌─────────────────────────────────────────────────────┐")
        print("│               📦 REPOSITORY                         │")
        print("│  (Permanent history)                                │")
        for c in self.commits[-3:]:
            head = " ← HEAD" if c["hash"] == self.HEAD else ""
            print(f"│    [{c['hash'][:7]}] {c['message'][:25]}{head:>10}  │"[:55] + "│")
        if not self.commits:
            print("│    (no commits yet)                                 │")
        print("└─────────────────────────────────────────────────────┘")
        print()

# Create global instance
git = GitSimulator()
print("✓ Git Simulator ready! Use 'git' to interact.")

✓ Git Simulator ready! Use 'git' to interact.


---
## The Three Areas

Git has three main areas:

1. **Working Directory** - Your actual files on disk
2. **Staging Area** - Files marked to go in the next commit
3. **Repository** - Permanent history of commits

Let's see this in action!

In [2]:
# Check initial state
git.status()


📍 Branch: main
📌 HEAD: None

📁 Working Directory:
   (empty)

📋 Staging Area:
   (empty)

📜 Commits: 0


In [6]:
# Create some files (like writing code)
git.create_file("app.py", "def main(): print('Hello World')")
git.create_file("README.md", "# My Project\nThis is a test project.")

✓ Created/modified: app.py
✓ Created/modified: README.md


In [4]:
# Check status - files are in working directory only
git.visualize()


════════════════════════════════════════════════════════════
                    GIT VISUALIZATION
════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────┐
│              📁 WORKING DIRECTORY                   │
│  (Your actual files on disk)                        │
│    • app.py           [modified]                ││
│    • README.md           [modified]                ││
└────────────────────────┬────────────────────────────┘
                         │
                    git add
                         ▼
┌─────────────────────────────────────────────────────┐
│               📋 STAGING AREA                       │
│  (Files ready to be committed)                      │
│    (empty)                                          │
└────────────────────────┬────────────────────────────┘
                         │
                   git commit
                         ▼
┌─────────────────────────────────────────────────────┐
│    

**→ Tell Claude: Where are the files? (Working directory only - not staged yet)**

---
## Staging Files (git add)

In [7]:
# Stage one file
git.add("app.py")
git.visualize()

✓ Staged: app.py

════════════════════════════════════════════════════════════
                    GIT VISUALIZATION
════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────┐
│              📁 WORKING DIRECTORY                   │
│  (Your actual files on disk)                        │
│    • app.py             [staged]                ││
│    • README.md           [modified]                ││
└────────────────────────┬────────────────────────────┘
                         │
                    git add
                         ▼
┌─────────────────────────────────────────────────────┐
│               📋 STAGING AREA                       │
│  (Files ready to be committed)                      │
│    • app.py                                       │
└────────────────────────┬────────────────────────────┘
                         │
                   git commit
                         ▼
┌─────────────────────────────────────────────

In [8]:
# Stage the other file
git.add("README.md")
git.visualize()

✓ Staged: README.md

════════════════════════════════════════════════════════════
                    GIT VISUALIZATION
════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────┐
│              📁 WORKING DIRECTORY                   │
│  (Your actual files on disk)                        │
│    • app.py             [staged]                ││
│    • README.md             [staged]                ││
└────────────────────────┬────────────────────────────┘
                         │
                    git add
                         ▼
┌─────────────────────────────────────────────────────┐
│               📋 STAGING AREA                       │
│  (Files ready to be committed)                      │
│    • app.py                                       │
│    • README.md                                    │
└────────────────────────┬────────────────────────────┘
                         │
                   git commit
                

**→ Notice: Files moved from working directory to staging area!**

---
## Creating a Commit (git commit)

In [9]:
# Create a commit
git.commit("Initial commit: add app and readme")
git.visualize()

✓ Committed: [99c15a9] Initial commit: add app and readme

════════════════════════════════════════════════════════════
                    GIT VISUALIZATION
════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────┐
│              📁 WORKING DIRECTORY                   │
│  (Your actual files on disk)                        │
│    • app.py           [modified]                ││
│    • README.md           [modified]                ││
└────────────────────────┬────────────────────────────┘
                         │
                    git add
                         ▼
┌─────────────────────────────────────────────────────┐
│               📋 STAGING AREA                       │
│  (Files ready to be committed)                      │
│    (empty)                                          │
└────────────────────────┬────────────────────────────┘
                         │
                   git commit
                         ▼
┌──

In [10]:
# View the log
git.log()


📜 Commit History:
──────────────────────────────────────────────────
  99c15a9 (HEAD)
  └─ Initial commit: add app and readme
     Files: ['app.py', 'README.md']



**→ Tell Claude: What happened to the staging area after commit? (It's empty!)**

---
## Making Changes and More Commits

In [11]:
# Modify a file
git.create_file("app.py", "def main(): print('Hello Kate!')")
git.visualize()

✓ Created/modified: app.py

════════════════════════════════════════════════════════════
                    GIT VISUALIZATION
════════════════════════════════════════════════════════════

┌─────────────────────────────────────────────────────┐
│              📁 WORKING DIRECTORY                   │
│  (Your actual files on disk)                        │
│    • app.py           [modified]                ││
│    • README.md           [modified]                ││
└────────────────────────┬────────────────────────────┘
                         │
                    git add
                         ▼
┌─────────────────────────────────────────────────────┐
│               📋 STAGING AREA                       │
│  (Files ready to be committed)                      │
│    (empty)                                          │
└────────────────────────┬────────────────────────────┘
                         │
                   git commit
                         ▼
┌─────────────────────────────────

In [13]:
# Stage and commit
git.add("app.py")
git.commit("Update greeting message")
git.log()

✓ Staged: app.py
✓ Committed: [1bccd24] Update greeting message

📜 Commit History:
──────────────────────────────────────────────────
  1bccd24 (HEAD)
  └─ Update greeting message
     Files: ['app.py']

  b18fff0
  └─ Update greeting message
     Files: ['app.py']

  99c15a9
  └─ Initial commit: add app and readme
     Files: ['app.py', 'README.md']



---
## Branches

Branches let you work on different versions of your code simultaneously.

Think of it as: "Let me try this idea without breaking what works"

In [14]:
# List branches
git.list_branches()


🌿 Branches:
  main * → 1bccd24


In [15]:
# Create a new branch for a feature
git.branch("feature/new-greeting")
git.list_branches()

✓ Created branch: feature/new-greeting

🌿 Branches:
  main * → 1bccd24
  feature/new-greeting → 1bccd24


In [16]:
# Switch to the new branch
git.checkout("feature/new-greeting")
git.list_branches()

✓ Switched to branch: feature/new-greeting

🌿 Branches:
  main → 1bccd24
  feature/new-greeting * → 1bccd24


In [17]:
# Make changes on the feature branch
git.create_file("app.py", "def main(): print('Hello Analytics Engineer!')")
git.add("app.py")
git.commit("Personalize greeting for target role")
git.log()

✓ Created/modified: app.py
✓ Staged: app.py
✓ Committed: [f3a4ef9] Personalize greeting for target role

📜 Commit History:
──────────────────────────────────────────────────
  f3a4ef9 (HEAD)
  └─ Personalize greeting for target role
     Files: ['app.py']

  1bccd24
  └─ Update greeting message
     Files: ['app.py']

  b18fff0
  └─ Update greeting message
     Files: ['app.py']

  99c15a9
  └─ Initial commit: add app and readme
     Files: ['app.py', 'README.md']



In [18]:
# Switch back to main
git.checkout("main")
git.status()

✓ Switched to branch: main

📍 Branch: main
📌 HEAD: 1bccd24

📁 Working Directory:
   app.py: def main(): print('Hello Kate!... 

📋 Staging Area:
   (empty)

📜 Commits: 4


**→ Tell Claude: What happened when we switched back to main? (Files reverted to main's version)**

---
## The Git Workflow Summary

```
1. Edit files         →  Working Directory
2. git add            →  Staging Area  
3. git commit         →  Repository
4. git push           →  Remote (GitHub)
```

For branches:
```
1. git branch name    →  Create branch
2. git checkout name  →  Switch to branch
3. (make commits)     →  Work on branch
4. git checkout main  →  Switch back
5. git merge name     →  Bring changes to main
```

---
## 🏋️ YOUR TURN: Exercise

Reset the simulator and practice:
1. Create a file called "analysis.py"
2. Stage and commit it
3. Create a branch called "feature/add-charts"
4. Switch to that branch
5. Add a file called "charts.py"
6. Commit it

In [ ]:
# Reset simulator
git = GitSimulator()

# Your code here


---
## 💡 Solution (Try first!)

In [ ]:
# Solution
git = GitSimulator()

# 1. Create file
git.create_file("analysis.py", "import pandas as pd")

# 2. Stage and commit
git.add("analysis.py")
git.commit("Add analysis script")

# 3. Create branch
git.branch("feature/add-charts")

# 4. Switch to branch
git.checkout("feature/add-charts")

# 5. Add file
git.create_file("charts.py", "import matplotlib.pyplot as plt")

# 6. Commit
git.add("charts.py")
git.commit("Add charts module")

# View result
git.log()
git.list_branches()

---
## 🎉 Module Complete!

**Key mental model:**
- **Working Directory** = your actual files
- **Staging Area** = "shopping cart" of changes to commit
- **Repository** = permanent history
- **Branches** = parallel versions of code
- **HEAD** = "you are here" pointer

**Common workflow:**
1. Edit files
2. `git add` files you want to commit
3. `git commit` to save to history
4. `git push` to share with others

**→ Tell Claude you're done and explain the three areas in your own words!**